In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 모델 초기화
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    temperature=0
)

In [ ]:
from typing import TypedDict, Annotated
import operator

from langchain_core.messages import HumanMessage, AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

# State 정의
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# Node 정의
def llm_node(state: MessagesState):
    response = model.invoke(
        state["messages"]
    )
    return {"messages": [response]}

# Graph 생성
graph_builder = StateGraph(MessagesState)

# Graph에 Node 추가
graph_builder.add_node("llm", llm_node)

# Edge 추가하여 Node 연결
graph_builder.add_edge(START, "llm")
graph_builder.add_edge("llm", END)

# Graph를 실행 가능한 형태로 컴파일
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

# Graph 실행
config = {"configurable": {"thread_id": "conversation_1"}}


In [3]:
# 내 이름 알려주기
human_message = HumanMessage(content="내 이름은 김일남이야.")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()


================================ Human Message =================================

내 이름은 김일남이야.
================================== Ai Message ==================================

안녕하세요, 김일남님. 만나서 반갑습니다!

무엇을 도와드릴까요?


In [4]:
# 내 이름 물어보기
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름은 김일남이야.
================================== Ai Message ==================================

안녕하세요, 김일남님. 만나서 반갑습니다!

무엇을 도와드릴까요?
================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

김일남님입니다.


In [5]:
# 스레드 변경
config = {"configurable": {"thread_id": "conversation_2"}}

In [6]:
# 내 이름 물어보기
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

저는 대규모 언어 모델이며, 이름이 없습니다.
